# 06c — Adaptation, extraction, and quantization — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/06.md) · [Course map](../PLAN.md) · [Project](../../projects/stage06/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Validate structured model outputs
- Explain and calculate low-rank updates
- Measure quantization error


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Adaptation, quantization, and source-grounded extraction

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

Pretraining learns general patterns from a large corpus. Supervised fine-tuning adapts a model using task examples. A frozen baseline leaves selected weights unchanged. LoRA trains low-rank update matrices while freezing a larger base weight; for a weight with input i, output o, and rank r, the update uses r(i+o) parameters instead of io.

Quantization stores numbers at reduced precision, introducing approximation error. Storage reduction is different from observed latency improvement. QLoRA combines a quantized base with trainable adapters and requires a compatible runtime; an INT8 arithmetic demonstration is not the same as GPU adapter training.

Structured extraction converts evidence into fields with an explicit schema. First locate source text, then normalize units, then validate. Unknown is an acceptable result when the source does not establish a field.

### Worked example: follow the values

For i=100, o=80, r=4, a dense weight has 8,000 entries; its LoRA pair has `4*(100+80)=720`, before other trainable components. A source saying '$2 million revenue' maps to 2,000,000 under an explicit USD convention. Preserve its exact quote and character offsets.

### Pause and explain

Why does a valid JSON object not prove an extraction is correct?

<details><summary>Check your reasoning after trying</summary>

The shape can be valid while its values, period, currency, or source attribution are wrong. Compare against the actual source.

</details>

### Common mistakes to check

Pin model identity and revision for reproducibility. Do not label a newly initialized classification head as a strong task-trained baseline.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

Pretraining learns broad patterns; supervised fine-tuning adjusts weights on task examples. Full fine-tuning updates all parameters. LoRA learns low-rank matrices A and B and adds a scaled BA update to a frozen weight matrix. This reduces trainable parameters but does not remove the base model’s inference cost. PEFT implements parameter-efficient approaches. QLoRA combines adapters with a quantized frozen base and requires compatible kernels/hardware.

Structured extraction requires a schema, numeric units, evidence spans, and failure handling. JSON syntax alone does not prove that a value is supported by the source. Validate both structure and semantics; keep an unknown result when evidence is absent.

Quantization maps weights to a smaller representation. FP32/FP16/BF16 differ in precision and range; INT8/INT4 use integer codes and scaling metadata. Weight storage reduction does not guarantee proportional speedup because kernels, activation memory, and KV cache also matter. Measure quality on held-out examples after adaptation or quantization.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import numpy as np
from pydantic import BaseModel, Field
print("A 1000×1000 matrix has", 1000*1000, "parameters")
print("Rank-8 adapters have", 8*(1000+1000), "parameters")


## Exercise 1: Validated extraction

Parse JSON with company (nonempty str) and revenue (finite float >=0), forbid extra keys. Return a validated model_dump dictionary.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def validate_extraction(text):
    from pydantic import BaseModel, Field, ConfigDict
    class Report(BaseModel):
        model_config = ConfigDict(extra="forbid")
        company: str = Field(min_length=1)
        revenue: float = Field(ge=0, allow_inf_nan=False)
    return Report.model_validate_json(text).model_dump()


In [ ]:
assert validate_extraction('{"company":"Example","revenue":100000000000}')["revenue"] == 1e11
from pydantic import ValidationError
expect_error(ValidationError, lambda: validate_extraction('{"company":"","revenue":-1}'))
print("Exercise 1: checks passed")


**Why this works:** The schema rejects malformed output; verifying the source and units is a separate check.


## Exercise 2: LoRA update

For W(out,in), A(rank,in), B(out,rank), return W+(alpha/rank)*(B@A).

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def lora_weight(w, a, b, alpha):
    return w + (alpha/a.shape[0])*(b@a)


In [ ]:
w = np.zeros((3, 4)); a = np.ones((2, 4)); b = np.ones((3, 2))
assert np.allclose(lora_weight(w, a, b, 2), np.full((3, 4), 2))
print("Exercise 2: checks passed")


**Why this works:** For low rank r, r(in+out) parameters can be much fewer than in×out.


## Exercise 3: Symmetric INT8 simulation

Return int8 codes, scale, and dequantized weights using max(abs(w))/127; all-zero arrays use scale=1.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def quantize(w):
    w = np.asarray(w, dtype=float)
    scale = float(np.max(np.abs(w)))/127 or 1.
    codes = np.clip(np.round(w/scale), -127, 127).astype(np.int8)
    return codes, scale, codes.astype(float)*scale


In [ ]:
codes, scale, restored = quantize([-.7, .1, 1.])
assert codes.dtype == np.int8
assert np.max(np.abs(restored-[-.7, .1, 1.])) <= scale/2+1e-12
assert np.allclose(quantize([0, 0])[2], 0)
print("Exercise 3: checks passed")


**Why this works:** This is a numerical simulation, not a fast quantized matrix-multiplication kernel.


## Independent transfer

Run the pretrained text-classification and PEFT lab in projects/stage06/pretrained.py after selecting/downloading a model. Compare frozen baseline versus fine-tuning on validation examples; record revision, license, cost, and test metrics. Audit extraction against source spans.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [06c interface and acceptance cases](../assignments/06c.md).
2. Copy the [blank starter](../assignments/starters/06c.py) to `work/assignments/06c.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 06c --solution work/assignments/06c.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/06c.md#06c-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 06c --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Does valid JSON imply truth?

   No. A valid schema can contain unsupported values.

2. What does LoRA change?

   It learns a low-rank weight update while keeping the original base weights frozen.


## Reading and review

- [Primary reference 1](https://huggingface.co/docs/peft/en/conceptual_guides/lora)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
